# iLSU-T (WhisperX) → SCULT Training (Colab)

Este notebook entrena **SCULT** usando datos iLSU-T que ya tenés (carpetas `episodes/` + `whisperx/`).

## Idea
1) Construir un dataset SLT a partir de WhisperX (clips + texto objetivo) con el pipeline del repo `lsu-pria`.
2) Exportar un bundle **SignJoey-compatible**.
3) Entrenar SCULT (repo de referencia) usando ese bundle.

## Requisitos
- Tener iLSU-T extraído en Drive, con estructura:
  - `.../ilsut_extracted/source2/episodes/` + `.../ilsut_extracted/source2/whisperx/`
  - `.../ilsut_extracted/source3/episodes/` + `.../ilsut_extracted/source3/whisperx/`
- Ejecutar en Colab con GPU (recomendado).


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Clone lsu-pria + install deps
# Si tu repo es privado o no está en GitHub, podés subirlo como ZIP a Colab y descomprimirlo en /content/lsu-pria.
LSU_PRIA_REPO_URL = ""  # @param {type:"string"}

!rm -rf /content/lsu-pria
if LSU_PRIA_REPO_URL.strip():
    !git clone -q "$LSU_PRIA_REPO_URL" /content/lsu-pria
else:
    print("LSU_PRIA_REPO_URL vacío: subí el repo como ZIP o setá la URL.")

%cd /content/lsu-pria

# Base deps (incluye scripts de dataset)
!python3 -m pip install -q -U pip
!python3 -m pip install -q -r requirements.txt

# Extras comunes para este flujo
!python3 -m pip install -q pandas pyyaml torchtext sacrebleu


In [ ]:
#@title Config — Paths + Parameters
import os
from pathlib import Path

# TODO: ajustá este path al lugar real en tu Drive
ILSUT_ROOT = "/content/drive/MyDrive/ilsut_extracted"  # @param {type:"string"}

# Dónde guardar el trabajo (clips + export + logs)
WORK_ROOT = "/content/drive/MyDrive/lsu-pria-runs/whisperx_slt_scult"  # @param {type:"string"}

SOURCES = ["source2", "source3"]

# Filtro de segmentos WhisperX (controla el tamaño del dataset)
MIN_WORDS = 1
MAX_WORDS = 60
MAX_CHARS = 240
MIN_DUR_MS = 700
MAX_DUR_MS = 25000
MAX_SEGMENTS_PER_EPISODE = 0  # 0 = sin límite; para pruebas usá 50-200

# Features (MediaPipe) → secuencia [T, F]
SAMPLE_FPS = 6.0
MAX_FRAMES = 48
PREPROCESS = True

# Export cap (0 = sin límite)
LIMIT = 0

print("ILSUT_ROOT:", ILSUT_ROOT)
print("WORK_ROOT:", WORK_ROOT)
Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)


In [ ]:
#@title Build WhisperX → SLT subset (clips + target_text)
subset_dir = Path(WORK_ROOT) / "subset"

cmd = f"""python3 scripts/prepare_whisperx_slt_subset.py \
  --root \"{ILSUT_ROOT}\" \
  --work-dir \"{subset_dir}\" \
  --sources {' '.join(SOURCES)} \
  --min-words {MIN_WORDS} \
  --max-words {MAX_WORDS} \
  --max-chars {MAX_CHARS} \
  --min-duration-ms {MIN_DUR_MS} \
  --max-duration-ms {MAX_DUR_MS} \
  --max-segments-per-episode {MAX_SEGMENTS_PER_EPISODE} \
  --export-clips \
  --clip-ext .mp4
"""
print(cmd)
!{cmd}


In [ ]:
#@title Export SignJoey bundle (neccam/slt format) from subset
export_dir = Path(WORK_ROOT) / "export"

cmd = f"""python3 scripts/export_ilsut_slt_dataset.py \
  --subset-dir \"{subset_dir}\" \
  --out-dir \"{export_dir}\" \
  --mode features \
  --sample-fps {SAMPLE_FPS} \
  --max-frames {MAX_FRAMES} \
  --clip-ext .mp4 \
  --backend neccam_slt \
  --limit {LIMIT} \
  {'--preprocess' if PREPROCESS else ''}
"""
print(cmd)
!{cmd}


In [ ]:
#@title Validate export
cmd = f"""python3 scripts/validate_ilsut_slt_dataset.py \
  --dataset-dir \"{export_dir}\" \
  --json-out \"{export_dir / 'dataset_validation.json'}\" \
  --md-out \"{export_dir / 'dataset_validation.md'}\" \
  --require-features
"""
print(cmd)
!{cmd}

print('Bundle path:', export_dir / 'backend' / 'neccam_slt' / 'data' / 'ilsut')
!ls -la "{export_dir}/backend/neccam_slt/data/ilsut" | head -n 50


## Train SCULT

Notas:
- SCULT (repo de referencia) está basado en configuraciones tipo SignJoey.
- Si el repo cambia estructura, ajustá los paths en las celdas siguientes.


In [ ]:
#@title Clone SCULT repo
!rm -rf /content/scult
!git clone -q https://github.com/avoskou/Stochastic-Transformer-Networks-with-Linear-Competing-Units-Application-to-end-to-end-SL-Translatio.git /content/scult
%cd /content/scult
!python3 -m pip install -q -U pip

# Instalar el repo (si expone signjoey)
!python3 -m pip install -q -e . || true

# Inspección rápida
!ls -la
!find . -maxdepth 3 -type f \( -name "sign.yaml" -o -name "*.yaml" -o -name "*.yml" \) | head -n 50


In [ ]:
#@title Copy exported dataset into SCULT data folder
from pathlib import Path
import shutil

SRC_DATA = Path(export_dir) / "backend" / "neccam_slt" / "data" / "ilsut"

# Destino: intentamos usar ./data/ilsut/ (común en repos SignJoey)
DST_DATA_ROOT = Path("/content/scult/data")
DST_ILSUT = DST_DATA_ROOT / "ilsut"
DST_ILSUT.mkdir(parents=True, exist_ok=True)

for fn in ["ilsut.train", "ilsut.val", "ilsut.test"]:
    src = SRC_DATA / fn
    dst = DST_ILSUT / fn
    print("copy", src, "->", dst)
    shutil.copy2(src, dst)

print("DST contents:")
!ls -la /content/scult/data/ilsut | head -n 50


In [ ]:
#@title Generate a SignJoey config for SCULT (patching configs/sign.yaml)
import time
from pathlib import Path

repo_root = Path('/content/scult')
configs_dir = repo_root / 'configs'
configs_dir.mkdir(parents=True, exist_ok=True)

# Try to find a base config
candidates = [
    repo_root / 'configs' / 'sign.yaml',
    repo_root / 'configs' / 'sign.yml',
]
base_cfg = next((p for p in candidates if p.exists()), None)
if base_cfg is None:
    found = list(repo_root.glob('**/sign.yaml')) + list(repo_root.glob('**/sign.yml'))
    base_cfg = found[0] if found else None
if base_cfg is None:
    raise RuntimeError('No base config found (expected configs/sign.yaml).')

cfg_txt = base_cfg.read_text(encoding='utf-8')
use_cuda = True
feature_size = 105  # el export de lsu-pria suele dar 105

def set_scalar(key: str, value: str) -> None:
    global cfg_txt
    lines = cfg_txt.splitlines()
    out = []
    replaced = False
    for ln in lines:
        if ln.lstrip().startswith(key + ':'):
            indent = ln[:len(ln) - len(ln.lstrip())]
            out.append(f"{indent}{key}: {value}")
            replaced = True
        else:
            out.append(ln)
    if not replaced:
        out.append(f"{key}: {value}")
    cfg_txt = "\n".join(out)

def set_section_scalar(section: str, key: str, value: str) -> None:
    global cfg_txt
    lines = cfg_txt.splitlines()
    out = []
    in_section = False
    replaced = False
    for ln in lines:
        if ln.startswith(section + ':'):
            in_section = True
            out.append(ln)
            continue
        if in_section and ln and not ln.startswith(' '):
            in_section = False
        if in_section and ln.lstrip().startswith(key + ':'):
            indent = ln[:len(ln) - len(ln.lstrip())]
            out.append(f"{indent}{key}: {value}")
            replaced = True
        else:
            out.append(ln)
    if not replaced:
        out2 = []
        inserted = False
        for ln in out:
            out2.append(ln)
            if ln.startswith(section + ':') and not inserted:
                out2.append(f"    {key}: {value}")
                inserted = True
        out = out2
    cfg_txt = "\n".join(out)

set_scalar('name', f"ilsut_scult_{int(time.time())}")

# data
set_section_scalar('data', 'data_path', repr(str(repo_root / 'data')))
set_section_scalar('data', 'train', 'ilsut/ilsut.train')
set_section_scalar('data', 'dev', 'ilsut/ilsut.val')
set_section_scalar('data', 'test', 'ilsut/ilsut.test')
set_section_scalar('data', 'feature_size', str(int(feature_size)))
set_section_scalar('data', 'level', 'word')
set_section_scalar('data', 'txt_lowercase', 'true')
set_section_scalar('data', 'max_sent_length', '400')

# training
set_section_scalar('training', 'use_cuda', 'true' if use_cuda else 'false')
set_section_scalar('training', 'overwrite', 'true')

out_cfg = configs_dir / 'ilsut_scult.yaml'
out_cfg.write_text(cfg_txt + "\n", encoding='utf-8')
print('Wrote:', out_cfg)
print('Base :', base_cfg)
!sed -n '1,120p' /content/scult/configs/ilsut_scult.yaml


In [ ]:
#@title Train (SignJoey entrypoint)
# Si falla, revisá primero que el repo instaló el módulo `signjoey`.
%cd /content/scult

!python3 -c "import signjoey; print('signjoey ok:', signjoey.__file__)" || true

# Entrenamiento
!python3 -m signjoey train configs/ilsut_scult.yaml
